# Tutorial 4 — Advanced concepts and chaining modes

So far we've used a block-scale urban model. This tutorial switches to a
single new building surrounded by existing neighbours, and shows two
different use cases:

- **Part A — Annual irradiance on roofs:** carve voxels that block solar
  radiation from reaching neighbouring rooftops (mode: `irradiance`)
- **Part B — Daylight on windows:** carve voxels that block daylight
  angles to neighbouring windows (mode: `daylight`)

Each part uses a different test surface: roofs for Part A, windows for Part B. Here the outputs of Part A are used as inputs for Part B


## Setup

In [ ]:
from pathlib import Path
import trimesh

from urbansolarcarver import user_config, run_pipeline


## Input Geometry Preview

The same two inputs as previous notebooks:

- **Max volume** (grey) -- the starting buildable volume. 
- **Neighbor_roofs** (red) -- surfaces where we want to protect solar access (for PVs and solar thermal).
- **Neighbor_windows** (blue) -- surfaces where we want to protect daylight access.

In [ ]:
#imports
REPO_ROOT = Path.cwd().parent
examples  = REPO_ROOT / "examples" / "test_inputs" / "new_building"

max_volume_path    = str(examples / "max_volume.ply")
neighbor_roofs     = str(examples / "neighbor_roofs.ply")
neighbor_windows   = str(examples / "neighbor_windows.ply")
epw_path           = str(ADDRESS_TO_EPW_FILE) #<-- ADD YOUR EPW FILE PATH HERE (donwload from ladybug.tools/epwmap/)
out_dir            = REPO_ROOT / "examples" / "test_outputs" / "tutorial_5"


In [ ]:
#preview geometry
max_vol = trimesh.load(max_volume_path)
roofs   = trimesh.load(neighbor_roofs)
windows = trimesh.load(neighbor_windows)

max_vol.visual.face_colors = [200, 200, 200, 100]   # grey, transparent
roofs.visual.face_colors   = [255, 100, 100, 255]   # red = roofs
windows.visual.face_colors = [100, 100, 255, 255]   # blue = windows

print(f"Max volume : {max_vol.faces.shape[0]} faces")
print(f"Roofs      : {roofs.faces.shape[0]} faces")
print(f"Windows    : {windows.faces.shape[0]} faces")

trimesh.Scene([max_vol, roofs, windows]).show(
    smooth=False, caption="New building (grey) + neighbour roofs (red) + windows (blue)"
)


## Part A — Annual Irradiance on Roofs

We want to know: **how much of the new building can we build without
blocking too much sunlight from neighbouring rooftops?**

Mode `irradiance` scores each voxel by how much solar radiation (kWh/m²)
it blocks on the test surfaces — in this case, the neighbour roofs.

We'll analyse the full year to capture annual irradiance.


In [ ]:
# Configure irradiance run for roofs

cfg_roofs = user_config(
    max_volume_path   = max_volume_path,
    test_surface_path = neighbor_roofs,          # ← roofs as test surface
    epw_path          = epw_path,
    out_dir           = str(out_dir / "partA_irradiance_roofs"),

    mode = "irradiance",

    # Full year
    start_month = 1, start_day = 1, start_hour = 0,
    end_month  = 12, end_day  = 31, end_hour  = 23,

    voxel_size     = 0.5,
    grid_step      = 0.5,
    threshold      = "carve_fraction",
    carve_fraction = 0.9,
    carve_above = True,                      # ← this ensures that stray islands of voxels above main carved volume are removed
    carve_above_min_consecutive = 3          # ← How many consecutive voxels above the first carved voxel to detect before removing everything above
)

In [ ]:
# Run the full pipeline

result_roofs = run_pipeline(cfg_roofs, out_dir / "partA_irradiance_roofs")
print("Part A done — irradiance envelope carved.")


In [ ]:
# View results

mesh_roofs = trimesh.load(str(result_roofs.export_path))
mesh_roofs.visual.face_colors = [255, 180, 100, 180]   # orange
roofs_ctx = trimesh.load(neighbor_roofs)
roofs_ctx.visual.face_colors = [255, 100, 100, 255]

print(f"Irradiance envelope: {mesh_roofs.faces.shape[0]} faces, {mesh_roofs.volume:.1f} m³")
trimesh.Scene([mesh_roofs, roofs_ctx]).show(
    smooth=False, caption="Part A: Irradiance envelope (orange) + roofs (red)"
)

---
## Part B — Daylight on Windows

Now we ask: **how much can we build without blocking daylight from
reaching neighbouring windows?**

Mode `daylight` uses the CIE standard sky dome to score each voxel by
how much sky visibility it blocks from the test surfaces (windows).


In [ ]:
# Part B config: daylight on windows

cfg_windows = user_config(
    max_volume_path   = str(result_roofs.export_path),   # ← IMPORTANT: Use part A output as input!
    test_surface_path = neighbor_windows,
    out_dir           = str(out_dir / "partB_daylight_windows"),

    mode = "daylight",

    voxel_size     = 0.5,
    grid_step      = 0.5,
    threshold      = "carve_fraction",
    carve_fraction = 0.7,
    carve_above = True,
    carve_above_min_consecutive = 3,
    apply_smoothing = True,            # ← we apply smoothing to reduce stray voxels and get a better mesh
    smooth_iters = 2,                  # ← conservative smoothing
)

In [ ]:
#  Run the full pipeline again

result_windows = run_pipeline(cfg_windows, out_dir / "partB_daylight_windows")
print("Part B done — daylight envelope carved.")


In [ ]:
# View the daylight envelope

mesh_windows = trimesh.load(str(result_windows.export_path))
mesh_windows.visual.face_colors = [100, 180, 255, 180]   # blue
windows_ctx = trimesh.load(neighbor_windows)
windows_ctx.visual.face_colors = [100, 100, 255, 255]
roofs_ctx = trimesh.load(neighbor_roofs)
roofs_ctx.visual.face_colors = [255, 100, 100, 255]

print(f"Daylight envelope: {mesh_windows.faces.shape[0]} faces, {mesh_windows.volume:.1f} m³")
trimesh.Scene([mesh_windows, roofs_ctx, windows_ctx]).show(
    smooth=False, caption="Part A+B: Envelope (blue) + roofs (red) + windows (dark blue)"
)
